<a href="https://colab.research.google.com/github/Balramt/PyTorch/blob/main/experiments/datasets_%26_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch Datasets and Data Loaders Classes


In [1]:
import torch
print(torch.__version__)

2.10.0+cpu


In [2]:
from sklearn.datasets import make_classification
import torch

In [3]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [4]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [5]:
X.shape

(10, 2)

In [6]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [7]:
y.shape

(10,)

In [8]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [9]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [10]:
X.shape

torch.Size([10, 2])

In [20]:
X.shape[0], X.shape[1], len(X)

(10, 2, 10)

In [11]:
y

tensor([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [12]:
from torch.utils.data import Dataset, DataLoader

In [22]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [23]:
dataset = CustomDataset(X, y)

In [24]:
len(dataset)

10

In [26]:
dataset[1]

(tensor([-1.1402, -0.8388]), tensor(0))

Now we will go for dataloader

In [31]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [32]:
for batch_features, batch_labels in dataloader:

  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[ 1.8997,  0.8344],
        [ 1.7273, -1.1858]])
tensor([1, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.9382, -0.5430]])
tensor([0, 1])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-0.5872, -1.9717]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------


We implement here the mini batched gradient descendent of previously implemented code

In [61]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [62]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [63]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [64]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [65]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [66]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [67]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [98]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [99]:
X_train_tensor.shape,X_train_tensor.shape[1]

(torch.Size([455, 30]), 30)

In [70]:
y_train_tensor.shape

torch.Size([455])

In [100]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [101]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [102]:
test_dataset[10], len(test_dataset)

((tensor([ 0.7306, -0.0831,  0.6942,  0.6296, -0.5532, -0.2372, -0.0674,  0.4375,
           0.3014, -1.0619,  0.7297,  0.3508,  0.6476,  0.5960, -0.2946, -0.0444,
          -0.1995,  0.6756,  0.0598, -0.1153,  0.7755,  0.1166,  0.6932,  0.6621,
          -0.7064, -0.2814, -0.2161,  0.4848, -0.1689, -0.6646]),
  tensor(1.)),
 114)

In [103]:
len(train_dataset)

455

In [104]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

Defining The Model

In [105]:
import torch.nn as nn

class MySimpleNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

Training Pipeline

In [106]:
learning_rate = 0.01
epochs = 25

In [107]:
# Create Model

model5 = MySimpleNN(X_train_tensor.shape[1])

# Define optimizers

optimizers = torch.optim.SGD(model5.parameters(), lr=learning_rate)

# Define loss function

loss_function = nn.BCELoss()

In [109]:
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model5(batch_features)

    # loss calculation
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # Clearing gradienets

    optimizers.zero_grad()

    # backward pass
    loss.backward()

    # Update parameters

    optimizers.step()

  print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 0.06314308196306229
Epoch: 2, Loss: 0.205600768327713
Epoch: 3, Loss: 0.053804729133844376
Epoch: 4, Loss: 0.16631551086902618
Epoch: 5, Loss: 0.1011393591761589
Epoch: 6, Loss: 0.09377558529376984
Epoch: 7, Loss: 0.05796337127685547
Epoch: 8, Loss: 0.06396825611591339
Epoch: 9, Loss: 0.4877167046070099
Epoch: 10, Loss: 0.038912031799554825
Epoch: 11, Loss: 0.07821222394704819
Epoch: 12, Loss: 0.0877525731921196
Epoch: 13, Loss: 0.050886452198028564
Epoch: 14, Loss: 0.0708867534995079
Epoch: 15, Loss: 0.09339187294244766
Epoch: 16, Loss: 0.041005659848451614
Epoch: 17, Loss: 0.24500353634357452
Epoch: 18, Loss: 0.02558809332549572
Epoch: 19, Loss: 0.04244367405772209
Epoch: 20, Loss: 0.26008912920951843
Epoch: 21, Loss: 0.15212757885456085
Epoch: 22, Loss: 0.12670551240444183
Epoch: 23, Loss: 0.026575712487101555
Epoch: 24, Loss: 0.25770312547683716
Epoch: 25, Loss: 0.13318222761154175


In [113]:
# Model evaluations

model5.eval() # set model to evaluate
accuracy_list = []
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model5(batch_features)
    y_pred = (y_pred > 0.5).float()

    # calculate the accuracy for current batch
    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

# calculate overall acuracy

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Overall Accuracy: {overall_accuracy}")

Overall Accuracy: 0.9704861044883728
